In [ ]:
from cellsweep.simulation import simulate_cells
from cellsweep.simulation import simple_simulation
import cellsweep.utils as cs_utils
from cellsweep import denoise_count_matrix
import scanpy as sc
import matplotlib.pyplot as plt
import matplotlib
import numpy as np
import os

cellsweep_dir = os.path.dirname(os.path.abspath(""))
out_dir = os.path.join(cellsweep_dir, "notebooks", "output", "debug_sim")

In [ ]:
%matplotlib widget

In [ ]:
data = simulate_cells(G=10000, N=8000, empty_prob=0.8)

In [ ]:
cs_utils.knee_plot(data)

In [ ]:
adata_cellmender_path = os.path.join(out_dir, "adata_cellmender.h5ad")
adata_log_path = os.path.join(out_dir, "cellsweep.log")


denoised_data = denoise_count_matrix(data, verbose=2, threads=8, adata_out = adata_cellmender_path, log_file = adata_log_path)

In [ ]:
sc.pp.pca(data, n_comps=30)
sc.pp.neighbors(data, n_neighbors=30, n_pcs=30, knn=True)
sc.tl.umap(data)
fig, ax = plt.subplots(figsize=(10, 7))
sc.pl.umap(data, color='celltype', ax=ax)
fig, ax = plt.subplots(figsize=(10, 7))
data.obs['log_ambient'] = np.log(data.obs['ambient_fraction'])
sc.pl.umap(data, color='log_ambient', ax=ax)

In [ ]:
real_mask = ~denoised_data.obs["is_empty"]
print(denoised_data.obs[real_mask])

In [ ]:
data.obs['percent_ambient_prediction_error'] = (data.obs['ambient_fraction'] - denoised_data.obs['alpha_hat'])/data.obs['ambient_fraction']
data.obs['percent_ambient_prediction_error'] = data.obs['percent_ambient_prediction_error'].mask(~real_mask, np.nan)
data.obs['predicted_type'] = denoised_data.obs['z_hat'].astype('category')
data.obs['prediction_error'] = (denoised_data.obs['z_hat'].astype(int) == denoised_data.obs['cellid'].astype(int))

fig, ax = plt.subplots(figsize=(10, 7))
sc.pl.umap(data, color='predicted_type', ax=ax)
fig, ax = plt.subplots(figsize=(10, 7))
sc.pl.umap(data, color='prediction_error',ax=ax)
fig, ax = plt.subplots(figsize=(10, 7))
sc.pl.umap(data, color='percent_ambient_prediction_error', ax=ax)